In [1]:
# Torchvision is a PyTorch library specifically designed for computer vision tasks.
import torchvision

### Transforms on Images

In [2]:
from torchvision.datasets import CIFAR10

In [3]:
import torchvision.transforms as transforms

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(), # It converts images into tensor format and scale between (0,1)
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)), # Normalize the data between (-1,1)
])

### Dataset

In [5]:
trainset = CIFAR10(root = "./data" , train = True , download = True , transform = transform)
testset = CIFAR10(root = "./data" , train = False , download = True , transform = transform )

### Data Loaders

In [6]:
from torch.utils.data import DataLoader

In [7]:
trainloader = DataLoader(trainset , batch_size = 64 , shuffle = True)
testloader = DataLoader(testset , batch_size = 64)

# CNN

### Model Definition

In [8]:
import torch.nn as nn

In [21]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN , self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3 , 32 , kernel_size = 3 ,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # Kernel size & Stride value

            nn.Conv2d(32 , 64 , kernel_size = 3 ,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # Kernel size & Stride value

            nn.Conv2d(64 , 128 , kernel_size = 3 ,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # Kernel size & Stride value
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) #Flattening
        x = self.fc_layers(x)

        return x 

In [22]:
model = CNN()

### Loss function and Optimizer

In [25]:
import torch.optim as optim
import torch

crietrion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [ ]:
epochs = 10 
train_losses = []
val_losses = []
best_loss = float("inf")

for epoch in range(epochs):
    model.train()

    running_train_loss = 0.0

    for image,label in trainloader:
        optimizer.zero_grad()
        
        outputs = model(image)
        loss = crietrion(outputs,label)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_epoch_loss = running_train_loss / len(trainloader)
    train_losses.append(avg_train_epoch_loss)

    model.eval()

    running_val_loss = 0.0

    with torch.no_grad():
        for image,label in testloader:
            outputs = model(image)
            loss = crietrion(outputs,label)

            running_val_loss +=loss.item()

    avg_val_epoch_loss = running_val_loss / len(testloader)
    val_losses.append(avg_val_epoch_loss)

    print(f"epoch : {epoch+1} training loss = {avg_train_epoch_loss}  validation loss = {avg_val_epoch_loss}")

    if avg_val_epoch_loss < best_loss:
        best_loss = avg_val_epoch_loss
        torch.save(model.state_dict(),"best_cnn_model.pt")

### Evaluation

In [28]:
model.load_state_dict(torch.load("best_cnn_model.pt"))

<All keys matched successfully>

In [29]:
model.eval()
correct_labels = 0
total_labels = 0

with torch.no_grad():
    for images , labels in testloader:
        outputs = model(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)


print(f"Accuracy : {(correct_labels/total_labels) *100}")

Accuracy : 75.55
